## 09.05 机器翻译与数据集 参考答案


### 环境配置


In [1]:
import os
import sys
sys.path.insert(0, "..")
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    import pypto
    import torch
    from torch import nn
    from torch.nn import functional as F
    import torch_npu
    import logging

warnings.filterwarnings("ignore", message="Permission mismatch")
warnings.filterwarnings("ignore", message="TASK_QUEUE_ENABLE")
warnings.filterwarnings("ignore", message="Cannot create tensor")
logging.getLogger('torch_npu').setLevel(logging.WARNING)
logging.getLogger('matplotlib').setLevel(logging.WARNING)
import matplotlib.pyplot as plt

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
pypto.pypto_impl.DeviceInit()

### 练习 9.5.1

**题目：** 在 `load_data_nmt` 中尝试不同的 `num_examples` 参数值，对源语言和目标语言词表大小有何影响？

**解答：** `num_examples` 越大，参与词元化的序列越多，因此所获得的词表越大。

以下使用 `torch` 编程：



In [ ]:
from src.utils import (read_data_nmt, preprocess_nmt, tokenize_nmt, Vocab,
                       build_array_nmt, load_array)

def load_data_nmt(batch_size, num_steps, num_examples):
    text = preprocess_nmt(read_data_nmt())
    source, target = tokenize_nmt(text, num_examples)
    src_vocab = Vocab(source, min_freq=2, reserved_tokens=['<pad>', '<bos>', '<eos>'])
    tgt_vocab = Vocab(target, min_freq=2, reserved_tokens=['<pad>', '<bos>', '<eos>'])
    src_array, src_valid_len = build_array_nmt(source, src_vocab, num_steps)
    tgt_array, tgt_valid_len = build_array_nmt(target, tgt_vocab, num_steps)
    data_iter = load_array((src_array, src_valid_len, tgt_array, tgt_valid_len), batch_size)
    return data_iter, src_vocab, tgt_vocab

num_examples = [100, 200, 300, 400, 500, 600]
src_lens, tgt_lens = [], []
for n in num_examples:
    train_iter, src_vocab, tgt_vocab = load_data_nmt(batch_size=2, num_steps=8, num_examples=n)
    src_lens.append(len(src_vocab))
    tgt_lens.append(len(tgt_vocab))
print('Source vocab sizes:', src_lens)
print('Target vocab sizes:', tgt_lens)

使用 `PyPTO` 编程（使用 `src.utils` 中的 `load_data_nmt`）：



In [ ]:
from src.utils import load_data_nmt

num_examples = [100, 200, 300, 400, 500, 600]
src_lens, tgt_lens = [], []
for n in num_examples:
    train_iter, src_vocab, tgt_vocab = load_data_nmt(batch_size=2, num_steps=8, num_examples=n)
    src_lens.append(len(src_vocab))
    tgt_lens.append(len(tgt_vocab))
print('Source vocab sizes:', src_lens)
print('Target vocab sizes:', tgt_lens)

### 练习 9.5.2

**题目：** 中文和日语等语言没有单词边界指示符，单词级词元化仍然合适吗？

**解答：** 仍然合适。虽然中文和日语没有类似空格的分隔符，但可通过基于词典、规则或统计的分词方法对文本进行单词级划分。单词级词元化有助于更准确地确定语境中的词义（词义消歧），以及在信息检索中利用关键词进行文本匹配。单词级词元化是 NLP 中非常关键的一步。



---
## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#/](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/)

